# Uplift Modeling: Uber Multi-Treatment Promo Targeting

## The Problem

Uber has a **reactivation budget** and wants to bring back lapsed riders. Three promo options:

| Treatment | Cost per user |
|-----------|---------------|
| No promo (control) | $0 |
| $5 ride credit | $5 |
| $10 ride credit | $10 |
| Free ride | $15 |

**Question:** Which riders should get which offer to **maximize reactivation ROI**?

A standard churn model tells us *who will return* — but not *who will return **because of** the promo*. We need uplift modeling to identify **persuadables** and avoid wasting promos on sure things, lost causes, and sleeping dogs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.calibration import CalibratedClassifierCV

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Simulate Experimental Data

30K users randomly assigned to control or one of three promo arms. We engineer **heterogeneous treatment effects** reflecting real user archetypes:

| Archetype | Profile | Behavior |
|-----------|---------|----------|
| **Sure Things** | Heavy users, many lifetime rides | Return anyway (uplift ≈ 0) |
| **Persuadable (easy)** | Light, recent users | $5 credit is enough (uplift ≈ +0.15) |
| **Persuadable (hard)** | Light, lapsed users | Need $10+ (uplift +0.05 with $5, +0.20 with $10) |
| **Lost Causes** | Very lapsed (>180 days) | Won't return regardless (uplift ≈ 0) |
| **Sleeping Dogs** | Small group | Promo contact *reduces* return rate |

In [ ]:
n = 30000

last_ride_days_ago = np.concatenate([
    np.random.exponential(15, n // 3),   # recent
    np.random.exponential(60, n // 3),   # lapsed
    np.random.exponential(200, n - 2 * (n // 3))  # very lapsed
])
last_ride_days_ago = np.clip(last_ride_days_ago, 1, 365)

lifetime_rides = np.random.poisson(lam=np.where(
    last_ride_days_ago < 30, 80, np.where(last_ride_days_ago < 90, 25, 8)
), size=n).astype(float)

avg_trip_value = np.random.lognormal(mean=2.5, sigma=0.4, size=n)
city_tier = np.random.choice([1, 2, 3], size=n, p=[0.3, 0.45, 0.25])

treatment = np.random.choice(['control', '$5_credit', '$10_credit', 'free_ride'], size=n)

heavy_user = lifetime_rides > 50
light_recent = (lifetime_rides <= 50) & (last_ride_days_ago < 60)
light_lapsed = (lifetime_rides <= 50) & (last_ride_days_ago >= 60) & (last_ride_days_ago < 180)
very_lapsed = last_ride_days_ago >= 180
sleeping_dog = np.random.rand(n) < 0.05  # 5% of population

base_prob = np.where(
    heavy_user, 0.7,
    np.where(light_recent, 0.25,
    np.where(light_lapsed, 0.10,
    0.03))  # very lapsed
)
base_prob += (city_tier == 1) * 0.05 - (city_tier == 3) * 0.03

uplift_5 = np.where(
    sleeping_dog, -0.08,
    np.where(heavy_user, 0.01,
    np.where(light_recent, 0.15,
    np.where(light_lapsed, 0.05,
    0.01)))  # very lapsed
)

uplift_10 = np.where(
    sleeping_dog, -0.10,
    np.where(heavy_user, 0.02,
    np.where(light_recent, 0.18,
    np.where(light_lapsed, 0.20,
    0.02)))  # very lapsed
)

uplift_free = np.where(
    sleeping_dog, -0.12,
    np.where(heavy_user, 0.03,
    np.where(light_recent, 0.20,
    np.where(light_lapsed, 0.22,
    0.03)))  # very lapsed
)

prob = base_prob.copy()
prob = np.where(treatment == '$5_credit', base_prob + uplift_5, prob)
prob = np.where(treatment == '$10_credit', base_prob + uplift_10, prob)
prob = np.where(treatment == 'free_ride', base_prob + uplift_free, prob)
prob = np.clip(prob, 0.01, 0.99)

noise = np.random.normal(0, 0.03, n)
prob = np.clip(prob + noise, 0.01, 0.99)

returned = np.random.binomial(1, prob)

df = pd.DataFrame({
    'last_ride_days_ago': last_ride_days_ago,
    'lifetime_rides': lifetime_rides,
    'avg_trip_value': avg_trip_value,
    'city_tier': city_tier,
    'treatment': treatment,
    'returned': returned
})

print(f"Dataset: {len(df):,} users")
print(f"\nTreatment distribution:")
print(df['treatment'].value_counts())
print(f"\nReturn rate by treatment:")
print(df.groupby('treatment')['returned'].mean().round(3))

## The Standard Churn Model Approach (And Why It Fails)

A typical data science team would build a **churn/return prediction model**: predict P(return) and target the users most likely to return. This sounds logical but is fundamentally flawed for targeting.

In [ ]:
features = ['last_ride_days_ago', 'lifetime_rides', 'avg_trip_value', 'city_tier']

churn_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
churn_model.fit(df[features], df['returned'])
df['churn_score'] = churn_model.predict_proba(df[features])[:, 1]

df['churn_rank'] = pd.qcut(df['churn_score'], q=10, labels=False, duplicates='drop')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churn_profile = df.groupby('churn_rank').agg(
    avg_return_rate=('returned', 'mean'),
    avg_lifetime_rides=('lifetime_rides', 'mean'),
    avg_days_lapsed=('last_ride_days_ago', 'mean')
).reset_index()

axes[0].bar(churn_profile['churn_rank'], churn_profile['avg_return_rate'], color='steelblue')
axes[0].set_xlabel('Churn Model Decile (0=least likely to return)')
axes[0].set_ylabel('Actual Return Rate')
axes[0].set_title('Churn Model: Targets Users Who Return Anyway')

axes[1].bar(churn_profile['churn_rank'], churn_profile['avg_lifetime_rides'], color='coral')
axes[1].set_xlabel('Churn Model Decile')
axes[1].set_ylabel('Avg Lifetime Rides')
axes[1].set_title('Top Deciles = Heavy Users (Sure Things)')

plt.tight_layout()
plt.show()

top_decile = df[df['churn_rank'] == 9]
print(f"Churn model top decile:")
print(f"  Avg lifetime rides: {top_decile['lifetime_rides'].mean():.0f}")
print(f"  Avg days since last ride: {top_decile['last_ride_days_ago'].mean():.0f}")
print(f"  Return rate: {top_decile['returned'].mean():.1%}")
print(f"  → These are SURE THINGS — they'd return without a promo!")

## Why Uplift, Not Standard Prediction

The churn model optimizes for **P(Y=1)** — the probability of returning. But for targeting, we need:

$$\text{Uplift} = P(Y=1 \mid T=t) - P(Y=1 \mid T=\text{control})$$

The churn model's top-scoring users are **heavy riders who return anyway** (sure things). Sending them $10 credits is pure waste.

| Approach | Optimizes | Problem |
|----------|-----------|----------|
| Churn model | P(return) | Targets sure things |
| Uplift model | P(return \| treated) - P(return \| control) | Targets persuadables |

**Alternative ruled out:** Using the churn model for targeting optimizes for the *wrong objective*. It spends budget on users who don't need it and ignores persuadable users with moderate baseline return probability.

## T-Learner Uplift Model

Train **separate outcome models** for each treatment arm. For each user, predict the outcome under each treatment. Uplift = predicted outcome under treatment minus predicted outcome under control.

In [ ]:
models = {}
treatments = ['control', '$5_credit', '$10_credit', 'free_ride']

for t in treatments:
    mask = df['treatment'] == t
    model = GradientBoostingClassifier(
        n_estimators=150, max_depth=4, min_samples_leaf=50, random_state=42
    )
    model.fit(df.loc[mask, features], df.loc[mask, 'returned'])
    models[t] = model
    print(f"Trained model for '{t}': {mask.sum():,} users")

for t in treatments:
    df[f'pred_{t}'] = models[t].predict_proba(df[features])[:, 1]

for t in ['$5_credit', '$10_credit', 'free_ride']:
    df[f'uplift_{t}'] = df[f'pred_{t}'] - df['pred_control']

print("\nAverage predicted uplift by treatment:")
for t in ['$5_credit', '$10_credit', 'free_ride']:
    print(f"  {t}: {df[f'uplift_{t}'].mean():.4f}")

## User Segmentation: Persuadables, Sure Things, Lost Causes, Sleeping Dogs

In [ ]:
df['max_uplift'] = df[['uplift_$5_credit', 'uplift_$10_credit', 'uplift_free_ride']].max(axis=1)
df['best_treatment'] = df[['uplift_$5_credit', 'uplift_$10_credit', 'uplift_free_ride']].idxmax(axis=1).str.replace('uplift_', '')

def classify_user(row):
    if row['max_uplift'] < -0.02:
        return 'Sleeping Dog'
    elif row['pred_control'] > 0.6 and row['max_uplift'] < 0.05:
        return 'Sure Thing'
    elif row['pred_control'] < 0.08 and row['max_uplift'] < 0.05:
        return 'Lost Cause'
    elif row['max_uplift'] >= 0.05:
        return 'Persuadable'
    else:
        return 'Ambiguous'

df['user_type'] = df.apply(classify_user, axis=1)

print("User type distribution:")
type_counts = df['user_type'].value_counts()
for utype, count in type_counts.items():
    pct = count / len(df) * 100
    print(f"  {utype}: {count:,} ({pct:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {
    'Persuadable': '#2ecc71', 'Sure Thing': '#3498db',
    'Lost Cause': '#95a5a6', 'Sleeping Dog': '#e74c3c', 'Ambiguous': '#f39c12'
}

for utype in df['user_type'].unique():
    mask = df['user_type'] == utype
    axes[0].scatter(
        df.loc[mask, 'pred_control'], df.loc[mask, 'max_uplift'],
        alpha=0.15, s=8, label=utype, color=colors.get(utype, 'gray')
    )
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('P(return | control)')
axes[0].set_ylabel('Max Uplift')
axes[0].set_title('User Type Landscape')
axes[0].legend(markerscale=5)

type_order = ['Persuadable', 'Sure Thing', 'Lost Cause', 'Sleeping Dog', 'Ambiguous']
existing_types = [t for t in type_order if t in type_counts.index]
bars = axes[1].bar(
    existing_types,
    [type_counts[t] for t in existing_types],
    color=[colors[t] for t in existing_types]
)
axes[1].set_ylabel('Count')
axes[1].set_title('User Type Distribution')
axes[1].tick_params(axis='x', rotation=30)

persuadables = df[df['user_type'] == 'Persuadable']
if len(persuadables) > 0:
    best_tx_counts = persuadables['best_treatment'].value_counts()
    axes[2].pie(best_tx_counts.values, labels=best_tx_counts.index,
                autopct='%1.0f%%', startangle=90,
                colors=['#27ae60', '#2980b9', '#8e44ad'])
    axes[2].set_title('Best Treatment for Persuadables')

plt.tight_layout()
plt.show()

## Budget-Constrained Treatment Optimization

For each user, assign the treatment that maximizes **(uplift × expected revenue) − cost**, subject to a total budget constraint.

In [ ]:
treatment_costs = {'control': 0, '$5_credit': 5, '$10_credit': 10, 'free_ride': 15}
avg_revenue_per_reactivation = 50  # expected LTV from reactivated rider
total_budget = 50000

roi_scores = pd.DataFrame(index=df.index)
for t in ['$5_credit', '$10_credit', 'free_ride']:
    uplift_col = f'uplift_{t}'
    cost = treatment_costs[t]
    roi_scores[t] = (df[uplift_col] * avg_revenue_per_reactivation) - cost

roi_scores['control'] = 0

df['optimal_treatment'] = roi_scores.idxmax(axis=1)
df['optimal_roi'] = roi_scores.max(axis=1)

df_sorted = df.sort_values('optimal_roi', ascending=False).copy()
df_sorted['treatment_cost'] = df_sorted['optimal_treatment'].map(treatment_costs)
df_sorted['cumulative_cost'] = df_sorted['treatment_cost'].cumsum()

within_budget = df_sorted['cumulative_cost'] <= total_budget
df_sorted['budget_treatment'] = np.where(within_budget, df_sorted['optimal_treatment'], 'control')

n_treated = within_budget.sum()
actual_spend = df_sorted.loc[within_budget, 'treatment_cost'].sum()

print(f"Budget: ${total_budget:,}")
print(f"Users treated: {n_treated:,} / {len(df):,} ({n_treated/len(df):.1%})")
print(f"Actual spend: ${actual_spend:,.0f}")
print(f"\nOptimal treatment allocation:")
print(df_sorted.loc[within_budget, 'optimal_treatment'].value_counts())

## Strategy Comparison: No Promos vs. Blanket vs. Churn-Model vs. Uplift-Optimized

In [ ]:
def simulate_strategy(df, strategy_name, assignments):
    """Simulate outcomes for a treatment assignment strategy using predicted uplift."""
    expected_returns = df['pred_control'].copy()

    for t in ['$5_credit', '$10_credit', 'free_ride']:
        mask = assignments == t
        expected_returns.loc[mask] += df.loc[mask, f'uplift_{t}']

    expected_returns = expected_returns.clip(0, 1)
    total_returns = expected_returns.sum()
    costs = assignments.map(treatment_costs).sum()
    revenue = total_returns * avg_revenue_per_reactivation
    profit = revenue - costs
    roi = profit / costs if costs > 0 else float('inf')

    return {
        'strategy': strategy_name,
        'total_returns': total_returns,
        'cost': costs,
        'revenue': revenue,
        'profit': profit,
        'roi': roi
    }

# Strategy A: No promos
no_promo = simulate_strategy(df, 'No Promos', pd.Series('control', index=df.index))

# Strategy B: Blanket $10 to everyone
blanket_10 = simulate_strategy(df, 'Blanket $10', pd.Series('$10_credit', index=df.index))

# Strategy C: Churn-model targeting (top users by churn score get $10)
budget_for_10 = total_budget // 10
churn_assignments = pd.Series('control', index=df.index)
top_churn = df.nlargest(budget_for_10, 'churn_score').index
churn_assignments.loc[top_churn] = '$10_credit'
churn_target = simulate_strategy(df, 'Churn-Model Targeting', churn_assignments)

# Strategy D: Uplift-optimized
uplift_assignments = pd.Series('control', index=df.index)
for idx in df_sorted[within_budget].index:
    uplift_assignments.loc[idx] = df_sorted.loc[idx, 'optimal_treatment']
uplift_opt = simulate_strategy(df, 'Uplift-Optimized', uplift_assignments)

results = pd.DataFrame([no_promo, blanket_10, churn_target, uplift_opt])

print("Strategy Comparison")
print("=" * 85)
for _, row in results.iterrows():
    print(f"\n{row['strategy']}:")
    print(f"  Expected returns: {row['total_returns']:,.0f}")
    print(f"  Cost:     ${row['cost']:>10,.0f}")
    print(f"  Revenue:  ${row['revenue']:>10,.0f}")
    print(f"  Profit:   ${row['profit']:>10,.0f}")
    if row['cost'] > 0:
        print(f"  ROI:      {row['roi']:>10.1f}x")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bar_colors = ['#95a5a6', '#e74c3c', '#f39c12', '#2ecc71']

axes[0].bar(results['strategy'], results['profit'], color=bar_colors)
axes[0].set_ylabel('Profit ($)')
axes[0].set_title('Profit by Strategy')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(results['strategy'], results['cost'], color=bar_colors)
axes[1].set_ylabel('Cost ($)')
axes[1].set_title('Cost by Strategy')
axes[1].tick_params(axis='x', rotation=30)

roi_values = [r if r != float('inf') else 0 for r in results['roi']]
axes[2].bar(results['strategy'], roi_values, color=bar_colors)
axes[2].set_ylabel('ROI (x)')
axes[2].set_title('ROI by Strategy')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Qini Curve / Uplift Curve Evaluation

The **Qini curve** measures how well the uplift model identifies persuadable users. It plots the cumulative incremental effect as we target more users (ranked by predicted uplift). A good model front-loads uplift: the first users targeted have the highest incremental impact.

In [ ]:
def qini_curve(df, uplift_col, treatment_label, ax, color, label):
    """Compute and plot Qini curve for a single treatment vs. control."""
    subset = df[df['treatment'].isin([treatment_label, 'control'])].copy()
    subset['is_treated'] = (subset['treatment'] == treatment_label).astype(int)
    subset = subset.sort_values(uplift_col, ascending=False).reset_index(drop=True)

    n_total = len(subset)
    cum_treated = subset['is_treated'].cumsum()
    cum_control = (1 - subset['is_treated']).cumsum()
    cum_returns_treated = (subset['returned'] * subset['is_treated']).cumsum()
    cum_returns_control = (subset['returned'] * (1 - subset['is_treated'])).cumsum()

    safe_control = cum_control.replace(0, np.nan)
    qini = cum_returns_treated - cum_returns_control * (cum_treated / safe_control)
    qini = qini.fillna(0)

    fraction_targeted = np.arange(1, n_total + 1) / n_total
    ax.plot(fraction_targeted, qini, color=color, label=label, linewidth=2)

    random_qini = np.linspace(0, qini.iloc[-1], n_total)
    return qini, random_qini, fraction_targeted


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

treatments_to_eval = [
    ('$5_credit', 'uplift_$5_credit', '#27ae60'),
    ('$10_credit', 'uplift_$10_credit', '#2980b9'),
    ('free_ride', 'uplift_free_ride', '#8e44ad'),
]

for i, (t_label, uplift_col, color) in enumerate(treatments_to_eval):
    qini, random_qini, frac = qini_curve(df, uplift_col, t_label, axes[i], color, 'Uplift Model')
    axes[i].plot(frac, random_qini, 'k--', alpha=0.5, label='Random')

    # Qini by churn score (wrong model)
    subset_churn = df[df['treatment'].isin([t_label, 'control'])].copy()
    subset_churn['is_treated'] = (subset_churn['treatment'] == t_label).astype(int)
    subset_churn = subset_churn.sort_values('churn_score', ascending=False).reset_index(drop=True)

    n_c = len(subset_churn)
    cum_t_c = subset_churn['is_treated'].cumsum()
    cum_c_c = (1 - subset_churn['is_treated']).cumsum()
    cum_ret_t_c = (subset_churn['returned'] * subset_churn['is_treated']).cumsum()
    cum_ret_c_c = (subset_churn['returned'] * (1 - subset_churn['is_treated'])).cumsum()
    safe_c_c = cum_c_c.replace(0, np.nan)
    qini_churn = cum_ret_t_c - cum_ret_c_c * (cum_t_c / safe_c_c)
    qini_churn = qini_churn.fillna(0)
    frac_c = np.arange(1, n_c + 1) / n_c
    axes[i].plot(frac_c, qini_churn, color='red', alpha=0.6, linestyle=':', label='Churn Model', linewidth=2)

    axes[i].set_xlabel('Fraction of Population Targeted')
    axes[i].set_ylabel('Cumulative Incremental Returns')
    axes[i].set_title(f'Qini Curve: {t_label}')
    axes[i].legend()

plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Uplift model curve above random = model identifies persuadables")
print("  - Churn model curve near/below random = churn score doesn't find persuadables")
print("  - Steeper initial slope = better targeting (high-value users found first)")

## Key Takeaways

1. **Churn models optimize the wrong thing.** They predict P(return), not treatment effect. Top-scored users are sure things who'd return without any promo.

2. **Uplift models find persuadables.** By estimating the *incremental* impact of each treatment, we identify users whose behavior actually changes because of the intervention.

3. **User segmentation matters.** The four types — persuadables, sure things, lost causes, sleeping dogs — require fundamentally different strategies. Sleeping dogs should *never* be contacted.

4. **Budget-constrained optimization maximizes ROI.** By ranking users on (uplift × revenue) − cost and allocating within budget, the uplift strategy achieves higher returns at lower cost than blanket or churn-targeted approaches.

5. **Qini curves evaluate uplift models properly.** Standard metrics (AUC, accuracy) don't apply. Qini curves measure how well the model concentrates incremental impact in its top-ranked users.

6. **Uplift = operationalized HTE.** This notebook applies heterogeneous treatment effect estimates to a real targeting decision — it's the bridge from causal estimation to causal action.